In [3]:
import pandas as pd
import re
import logging
import warnings
from collections import defaultdict
from rapidfuzz import fuzz
from sqlalchemy import create_engine, text
from tqdm import tqdm

# =====================================================
# CONFIG & LOGGING
# =====================================================
pd.options.mode.chained_assignment = None
warnings.filterwarnings('ignore')

OUTPUT_PATH = r"C:/Users/Singhp37/OneDrive - Moody's/Desktop/Work/UK/UK_Output_Final.xlsx"
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

# =====================================================
# PATTERNS & LOOKUPS (FULL UK + ISLANDS)
# =====================================================
POSTCODE_RE = re.compile(r'([A-Z][A-HJ-Y]?\d[A-Z\d]?\s?\d[A-Z]{2}|GIR\s?0AA)', re.I)

COMMON_CITIES = [
    # --- ENGLAND ---
    'LONDON', 'BIRMINGHAM', 'LIVERPOOL', 'BRISTOL', 'MANCHESTER', 'LEEDS', 
    'SHEFFIELD', 'LEICESTER', 'COVENTRY', 'NOTTINGHAM', 'NEWCASTLE UPON TYNE', 
    'SOUTHAMPTON', 'DERBY', 'PORTSMOUTH', 'BRIGHTON', 'PLYMOUTH', 'WOLVERHAMPTON', 
    'NORWICH', 'YORK', 'CAMBRIDGE', 'OXFORD', 'STOKE-ON-TRENT', 'HULL', 'KINGSTON UPON HULL',
    'SUNDERLAND', 'BRADFORD', 'MILTON KEYNES', 'PETERBOROUGH', 'CHELMSFORD', 'COLCHESTER',
    'EXETER', 'GLOUCESTER', 'CHICHESTER', 'CANTERBURY', 'CARLISLE', 'CHESTER', 
    'DONCASTER', 'DURHAM', 'ELY', 'HEREFORD', 'LANCASTER', 'LICHFIELD', 'LINCOLN', 
    'PRESTON', 'RIPON', 'SALFORD', 'SALISBURY', 'ST ALBANS', 'TRURO', 'WAKEFIELD', 
    'WELLS', 'WESTMINSTER', 'WINCHESTER', 'WORCESTER', 'BATH', 'READING', 'NORTHAMPTON', 
    'LUTON', 'BLACKPOOL', 'SWINDON', 'WARRINGTON', 'HUDDERSFIELD', 'WATFORD', 'SLOUGH', 
    'BASILDON', 'MAIDSTONE', 'BROMLEY', 'CROYDON', 'GUILDFORD', 'HIGH WYCOMBE', 'STOCKPORT', 'MIDDLESBROUGH',
    # --- SCOTLAND ---
    'GLASGOW', 'EDINBURGH', 'ABERDEEN', 'DUNDEE', 'INVERNESS', 'PERTH', 
    'STIRLING', 'DUNFERMLINE', 'AYR', 'KILMARNOCK', 'FALKIRK', 'KIRKCALDY', 'MOTHERWELL',
    # --- WALES ---
    'CARDIFF', 'SWANSEA', 'NEWPORT', 'WREXHAM', 'BANGOR', 'ST ASAPH', 'ST DAVIDS', 
    'LLANELLI', 'BRIDGEND', 'BARRY', 'PONTYPRIDD',
    # --- NORTHERN IRELAND ---
    'BELFAST', 'LONDONDERRY', 'DERRY', 'LISBURN', 'NEWRY', 'ARMAGH', 'BALLYMENA', 'COLERAINE',
    # --- ISLANDS ---
    'DOUGLAS', 'SAINT HELIER', 'ST HELIER', 'SAINT PETER PORT', 'ST PETER PORT'
]

CITY_PC_PREFIXES = {
    'ABERDEEN': ['AB'], 'ARMAGH': ['BT'], 'BANGOR': ['LL', 'BT'], 'BATH': ['BA'],
    'BELFAST': ['BT'], 'BIRMINGHAM': ['B'], 'BRADFORD': ['BD'], 'BRIGHTON': ['BN'],
    'BRISTOL': ['BS'], 'CAMBRIDGE': ['CB'], 'CANTERBURY': ['CT'], 'CARDIFF': ['CF'],
    'CARLISLE': ['CA'], 'CHELMSFORD': ['CM'], 'CHESTER': ['CH'], 'CHICHESTER': ['PO'],
    'COLCHESTER': ['CO'], 'COVENTRY': ['CV'], 'DERBY': ['DE'], 'DERRY': ['BT'],
    'DONCASTER': ['DN'], 'DUNDEE': ['DD'], 'DUNFERMLINE': ['KY'], 'DURHAM': ['DH'],
    'EDINBURGH': ['EH'], 'ELY': ['CB'], 'EXETER': ['EX'], 'GLASGOW': ['G', 'PA', 'KA'],
    'GLOUCESTER': ['GL'], 'HEREFORD': ['HR'], 'HULL': ['HU'], 'INVERNESS': ['IV'],
    'KINGSTON UPON HULL': ['HU'], 'LANCASTER': ['LA'], 'LEEDS': ['LS', 'WF'],
    'LEICESTER': ['LE'], 'LICHFIELD': ['WS'], 'LINCOLN': ['LN'], 'LISBURN': ['BT'],
    'LIVERPOOL': ['L', 'CH', 'WA'], 'LONDON': ['E', 'EC', 'N', 'NW', 'SE', 'SW', 'W', 'WC', 'BR', 'CR', 'DA', 'EN', 'HA', 'IG', 'KT', 'RM', 'SM', 'TW', 'UB', 'WD'],
    'LONDONDERRY': ['BT'], 'MANCHESTER': ['M', 'SK', 'OL', 'WA', 'WN'], 'MILTON KEYNES': ['MK'],
    'NEWCASTLE UPON TYNE': ['NE'], 'NEWPORT': ['NP'], 'NEWRY': ['BT'], 'NORWICH': ['NR'],
    'NOTTINGHAM': ['NG'], 'OXFORD': ['OX'], 'PERTH': ['PH'], 'PETERBOROUGH': ['PE'],
    'PLYMOUTH': ['PL'], 'PORTSMOUTH': ['PO'], 'PRESTON': ['PR'], 'RIPON': ['HG'],
    'SALFORD': ['M'], 'SALISBURY': ['SP'], 'SHEFFIELD': ['S'], 'SOUTHAMPTON': ['SO'],
    'ST ALBANS': ['AL'], 'ST ASAPH': ['LL'], 'ST DAVIDS': ['SA'], 'STIRLING': ['FK'],
    'STOKE-ON-TRENT': ['ST'], 'SUNDERLAND': ['SR'], 'SWANSEA': ['SA'], 'TRURO': ['TR'],
    'WAKEFIELD': ['WF'], 'WELLS': ['BA'], 'WESTMINSTER': ['SW', 'W', 'WC'],
    'WINCHESTER': ['SO'], 'WOLVERHAMPTON': ['WV', 'DY'], 'WORCESTER': ['WR'],
    'WREXHAM': ['LL'], 'YORK': ['YO'], 'DOUGLAS': ['IM'], 'SAINT HELIER': ['JE'], 
    'ST HELIER': ['JE'], 'SAINT PETER PORT': ['GY'], 'ST PETER PORT': ['GY']
}

GEO_NOISE = re.compile(
    r'\b('
    r'uk|u k|u\.k\.?|u-k|united kingdom|the united kingdom|'
    r'britain|great britain|g\.b|g b|gb|'
    r'you kay|you\-kay|yo kay|yoo kay|yu kei|yukei|ukei|ukai|youkei|'
    r'royaume uni|grande bretagne|angleterre|regne uni|'
    r'reino unido|gran bretana|gran bretaña|'
    r'regno unito|gran bretagna|'
    r'vereinigt(?:es)? königreich|grossbritannien|vereinigtes koenigreich|'
    r'verenigd koninkrijk|storbritannien|forenede kongerige|storbritanien|'
    r'velikobrit|velikobritaniya|velikobritaniia|velikobritani|'
    r'великобритания|англия|британия|соединенное королевство|соед корол|'
    r'britaniya|bartaniya|bartaniyah|inglistan|inglistaan|'
    r'england|scotland|wales|northern ireland|'
    r'channel islands|jersey|isle of man'
    r')\b',
    re.I
)

LEGAL_RE = re.compile(r'\b(LTD|LIMITED|PLC|LLP|LLC|INC|CORP|CORPORATION|INCORPORATED)\b', re.I)
COMPANY_WORD_RE = re.compile(r'\b(COMPANY|CO|CO\.)\b', re.I)

LEGAL_CANONICAL = {
    'ltd': 'limited', 'limited': 'limited', 
    'plc': 'plc', 'llp': 'llp', 'llc': 'llc', 
    'inc': 'incorporated', 'incorporated': 'incorporated',
    'corp': 'corporation', 'corporation': 'corporation',
    '&': 'and'
}

ADDR_NORM = {
    'ST': 'STREET', 'RD': 'ROAD', 'AVE': 'AVENUE', 'CT': 'COURT', 'DR': 'DRIVE', 
    'LN': 'LANE', 'PL': 'PLACE', 'SQ': 'SQUARE', 'EST': 'ESTATE', 'IND': 'INDUSTRIAL'
}

# =====================================================
# LOGIC FUNCTIONS
# =====================================================

def get_legal_type(name):
    if not name: return "NONE"
    n = str(name).upper()
    priority_1 = ['LIMITED', 'LTD', 'PLC', 'LLP', 'LLC', 'INCORPORATED', 'INC']
    for p in priority_1:
        if re.search(r'\b' + p + r'\b', n):
            val = p.lower()
            return LEGAL_CANONICAL.get(val, val).upper()
    priority_2 = ['CORPORATION', 'CORP']
    for p in priority_2:
        if re.search(r'\b' + p + r'\b', n):
            val = p.lower()
            return LEGAL_CANONICAL.get(val, val).upper()
    return "NONE"

def is_incomplete_name(name):
    n = str(name).strip().upper()
    words = n.split()
    if len(words) < 2: return True
    if len(re.sub(r'[^A-Z]', '', words[0])) <= 1: return True
    return False

def apply_normalization(text, mapping):
    if not text: return ""
    t = str(text).upper().strip()
    for short, full in mapping.items():
        t = re.sub(r'\b' + re.escape(short) + r'\b', full.upper(), t)
    return ' '.join(t.split())

def parse_address_logic_v5(addr_raw, city_raw, pc_raw):
    a, c, p = str(addr_raw or "").upper(), str(city_raw or "").upper(), str(pc_raw or "").upper()
    master = f"{a} {c} {p}".replace("NONE", "").replace("NULL", "").strip()
    master = GEO_NOISE.sub("", master).strip()
    f_pc, f_city = "", ""
    pc_match = POSTCODE_RE.search(master)
    if pc_match:
        f_pc = pc_match.group(1).replace(" ", "")
        master = master.replace(pc_match.group(1), "").strip(", ")
    for city_name in COMMON_CITIES:
        if re.search(r'\b' + re.escape(city_name) + r'\b', master, re.I):
            f_city = city_name
            master = re.sub(r'\b' + re.escape(city_name) + r'\b', '', master, flags=re.I).strip(", ")
            break
    f_addr = apply_normalization(master, ADDR_NORM)
    return f_addr, f_city, f_pc

def clean_company_name(name):
    n = str(name).upper().replace('&', ' AND ')
    n = LEGAL_RE.sub('', n)
    n = COMPANY_WORD_RE.sub('', n)
    clean = ' '.join(re.sub(r'[^A-Z0-9 ]', ' ', n).split()).strip()
    return clean

# =====================================================
# MAIN ENGINE
# =====================================================
if __name__ == "__main__":
    engine = create_engine("mssql+pyodbc://@QIG-WXOBOSDB501.analytics.moodys.net/bvdaffils?driver=ODBC+Driver+17+for+SQL+Server&trusted_connection=yes")
    good_lookup, direct_list = defaultdict(list), []
    id_name_map, id_counts = {}, defaultdict(int)

    sql_query = """
    SELECT * FROM (
        SELECT Id, Name, Address, PCCity, PostCode 
        FROM Companies WITH (NOLOCK) 
        WHERE Id LIKE 'GBC*%' 
          AND (Name NOT LIKE '%JointVenture%' AND Name NOT LIKE '%JV%' AND Name NOT LIKE '%Joint%' AND Name NOT LIKE '%Venture%' 
          AND Name NOT LIKE '%Shareholder%' AND Name NOT LIKE '%TRUST%' AND Name NOT LIKE '%Trustee%' AND Name NOT LIKE '%NOMINE%' 
          AND Name NOT LIKE '%Fund%' AND Name NOT LIKE '%Account%' AND Name NOT LIKE '%Acc%' AND Name NOT LIKE '%A/C%' 
          AND Name NOT LIKE '%Branch%' AND Name NOT LIKE '%FOREIGN%' AND Name NOT LIKE '%Family%' AND Name NOT LIKE '%SHARE%' 
          AND Name NOT LIKE '%PENSION%' AND Name NOT LIKE '%DIRECTOR%' AND Name NOT LIKE '%Board%' AND Name NOT LIKE '%GP%' 
          AND Name NOT LIKE '%Partner%' AND Name NOT LIKE '%ADMIN%' AND Name NOT LIKE '%OVERSEAS%' AND Name NOT LIKE '%UNION%' 
          AND Name NOT LIKE '%CHARTERED%' AND Name NOT LIKE '%SURVEYORS%' AND Name NOT LIKE '%KOMPANIYA%' 
          AND Name NOT LIKE '%ASSOCIATION%' AND Name NOT LIKE '%SUBSIDIARY%' AND Name NOT LIKE '%T/A%' AND Name NOT LIKE '%C/O%')
          AND Person NOT IN ('I', 'L', 'K', 'D')
          AND(Branch IS NULL OR Branch IN (0, 3))
          AND ([Foreign] IS NULL OR [Foreign] IN (0, 3))
          AND Name NOT LIKE '%[0-9]%'
          AND Name NOT LIKE '%([A-Z])%' 
          AND Name NOT LIKE '% I %' AND Name NOT LIKE '% I'
          AND Name NOT LIKE '% II %' AND Name NOT LIKE '% II'
          AND Name NOT LIKE '% III %' AND Name NOT LIKE '% III'
          AND Name NOT LIKE '% IV %' AND Name NOT LIKE '% IV'
          AND Name NOT LIKE '% V %' AND Name NOT LIKE '% V'
          AND Name NOT LIKE '% X %' AND Name NOT LIKE '% X'
          AND Name NOT LIKE '% L %' AND Name NOT LIKE '% L'
          AND Name NOT LIKE '% C %' AND Name NOT LIKE '% C'
    ) AS D
    UNION ALL
    SELECT * FROM (
        SELECT Id, Name, Address, PCCity, PostCode 
        FROM Companies WITH (NOLOCK) 
        WHERE (Id LIKE 'GBC%' AND Id NOT LIKE 'GBC*%' AND Id NOT LIKE 'GBCOE%' AND Id NOT LIKE 'GBCBOE%'
          AND Id NOT LIKE 'GBCZEF%' AND Id NOT LIKE 'GBCLEI%' AND Id NOT LIKE 'GBC%-%')
          AND (Name NOT LIKE '%JointVenture%' AND Name NOT LIKE '%JV%' AND Name NOT LIKE '%Joint%' AND Name NOT LIKE '%Venture%' 
          AND Name NOT LIKE '%Shareholder%' AND Name NOT LIKE '%TRUST%' AND Name NOT LIKE '%Trustee%' AND Name NOT LIKE '%NOMINE%' 
          AND Name NOT LIKE '%Fund%' AND Name NOT LIKE '%Account%' AND Name NOT LIKE '%Acc%' AND Name NOT LIKE '%A/C%' 
          AND Name NOT LIKE '%Branch%' AND Name NOT LIKE '%FOREIGN%' AND Name NOT LIKE '%Family%' AND Name NOT LIKE '%SHARE%' 
          AND Name NOT LIKE '%PENSION%' AND Name NOT LIKE '%DIRECTOR%' AND Name NOT LIKE '%Board%' AND Name NOT LIKE '%GP%' 
          AND Name NOT LIKE '%Partner%' AND Name NOT LIKE '%ADMIN%' AND Name NOT LIKE '%OVERSEAS%' AND Name NOT LIKE '%UNION%' 
          AND Name NOT LIKE '%CHARTERED%' AND Name NOT LIKE '%SURVEYORS%' AND Name NOT LIKE '%KOMPANIYA%' 
          AND Name NOT LIKE '%ASSOCIATION%' AND Name NOT LIKE '%SUBSIDIARY%' AND Name NOT LIKE '%T/A%' AND Name NOT LIKE '%C/O%')
          AND Person NOT IN ('I', 'L', 'K', 'D')
          AND(Branch IS NULL OR Branch IN (0, 3))
          AND ([Foreign] IS NULL OR [Foreign] IN (0, 3))
          AND Name NOT LIKE '%[0-9]%'
          AND Name NOT LIKE '%([A-Z])%' 
          AND Name NOT LIKE '% I %' AND Name NOT LIKE '% I'
          AND Name NOT LIKE '% II %' AND Name NOT LIKE '% II'
          AND Name NOT LIKE '% III %' AND Name NOT LIKE '% III'
          AND Name NOT LIKE '% IV %' AND Name NOT LIKE '% IV'
          AND Name NOT LIKE '% V %' AND Name NOT LIKE '% V'
          AND Name NOT LIKE '% X %' AND Name NOT LIKE '% X'
          AND Name NOT LIKE '% L %' AND Name NOT LIKE '% L'
          AND Name NOT LIKE '% C %' AND Name NOT LIKE '% C'
    ) AS G
    """

    logging.info("Step 1: Fetching Data...")
    df = pd.read_sql(text(sql_query), engine)
    
    for r in df.itertuples(index=False):
        p_addr, p_city, p_pc = parse_address_logic_v5(r.Address, r.PCCity, r.PostCode)
        clean_n = clean_company_name(r.Name)
        clean_id = str(r.Id).replace('!', '').replace('*', '')
        id_counts[clean_id] += 1
        if '!' not in str(r.Id): id_name_map[clean_id] = r.Name

        row = {
            'Id': r.Id, 'Name': r.Name, 'clean': clean_n,
            'Parsed_Addr': p_addr, 'Parsed_City': p_city, 'Parsed_PC': p_pc,
            'Orig_Addr': r.Address, 'Orig_City': r.PCCity, 'Orig_PC': r.PostCode,
            'Legal': get_legal_type(r.Name),
            'Has_Company': bool(COMPANY_WORD_RE.search(str(r.Name)))
        }
        if str(r.Id).startswith('GBC*'): direct_list.append(row)
        else:
            if clean_n: good_lookup[clean_n].append(row)

    sheets_data = [[] for _ in range(9)]
    sheet_names = ['1_SameName_Legal_NoAddr', '2_SameName_Legal_SameAddr', '3_SameName_SameLegal_DiffAddr', '4_Without_Legal_Type_Matches', '5_CO_Risk_Matches', '6_SameName_DiffLegal_NoAddr', '7_SameName_SameAddr_DiffLegal', '8_Multiple Match', '9_No Match']

    for d in tqdm(direct_list, desc="Matching"):
        if is_incomplete_name(d['Name']): continue
        raw_candidates = good_lookup.get(d['clean'], [])
        
        # SYMMETRY CHECK: Both sides must be equal on 'Company' presence
        candidates = [c for c in raw_candidates if d['Has_Company'] == c['Has_Company']]

        if not candidates:
            sheets_data[8].append({'Direct ID': d['Id'], 'Direct Name': d['Name'], 'Direct Legal': d['Legal']})
            continue

        if len(candidates) > 1:
            for c in candidates:
                sheets_data[7].append({'Direct ID': d['Id'], 'Direct Name': d['Name'], 'Direct Legal': d['Legal'], 'Good ID': c['Id'], 'Good Name': c['Name'], 'Good Legal': c['Legal']})
            continue

        g = candidates[0]
        co_risk_pattern = re.compile(r'\b(CO|CO\.|& CO|AND CO)\b', re.I)
        is_co_risk = bool(co_risk_pattern.search(str(d['Name'])) or co_risk_pattern.search(str(g['Name'])))

        name_score = fuzz.token_sort_ratio(d['clean'], g['clean'])
        addr_score = fuzz.token_set_ratio(d['Parsed_Addr'], g['Parsed_Addr'])
        if d['Parsed_PC'] == g['Parsed_PC'] and d['Parsed_PC'] != "": addr_score = 100

        same_legal = (d['Legal'] == g['Legal'])
        d_no_addr = not (d['Parsed_Addr'] or d['Parsed_City'] or d['Parsed_PC'])
        g_clean_id, d_clean_id = str(g['Id']).replace('!', ''), str(d['Id']).replace('*', '').replace('!', '')

        res = {
            'Direct ID': d['Id'], 
            'Direct Name': d['Name'], 
            'Direct Legal': d['Legal'],
            'Direct ID Link Count': id_counts.get(d_clean_id, 0),
            'Good ID': g['Id'], 
            'Good Name': g['Name'], 
            'Good Legal': g['Legal'],
            'New ID': g_clean_id, 
            'New Name': id_name_map.get(g_clean_id, g['Name']),
            'New ID Link Count': id_counts.get(g_clean_id, 0),
            'D_Orig_Addr': d['Orig_Addr'], 
            'G_Orig_Addr': g['Orig_Addr'],
            'D_Orig_City': d['Orig_City'], 
            'G_Orig_City': g['Orig_City'],
            'D_Orig_PC': d['Orig_PC'], 
            'G_Orig_PC': g['Orig_PC'],
            'D_Parsed_Addr': d['Parsed_Addr'],
            'G_Parsed_Addr': g['Parsed_Addr'],
            'D_Parsed_PC': d['Parsed_PC'], 
            'G_Parsed_PC': g['Parsed_PC'], 
            'Name_Fuzzy_Score': name_score, 
            'Addr_Fuzzy_Score': addr_score
        }

        if is_co_risk: sheets_data[4].append(res) 
        elif d['Legal'] == "NONE" or g['Legal'] == "NONE": sheets_data[3].append(res)
        elif same_legal and d_no_addr: sheets_data[0].append(res)
        elif same_legal and addr_score >= 65: sheets_data[1].append(res)
        elif same_legal and addr_score < 65 and not d_no_addr: sheets_data[2].append(res)
        elif not same_legal and d_no_addr: sheets_data[5].append(res)
        elif not same_legal and addr_score >= 65: sheets_data[6].append(res)
        else: sheets_data[8].append(res)

    with pd.ExcelWriter(OUTPUT_PATH, engine='xlsxwriter') as writer:
        for data, name in zip(sheets_data, sheet_names):
            
            if data: pd.DataFrame(data).to_excel(writer, sheet_name=name, index=False)

    logging.info(f"Done!")

2026-03-04 01:14:22,141 - INFO - Step 1: Fetching Data...
Matching: 100%|████████████████████████████████████████████████████████████████| 88211/88211 [00:11<00:00, 7464.71it/s]
2026-03-04 03:50:23,169 - INFO - Done!
